# House Prices：从 Pandas 到预测
先按同目录 README 获取数据。Notebook 可从仓库根目录或本目录打开；完整模型流程复用 train.py，避免两份代码漂移。

In [ ]:
from pathlib import Path
import os

root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "pyproject.toml").exists():
    raise RuntimeError("请从仓库内打开 Notebook")
os.chdir(root)
import sys
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import pandas as pd
import numpy as np

DATA = root / "data/house_prices"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print(train.shape, test.shape)
train.head()

## 类型与缺失
先看数据字典，不把高缺失率直接等同于无用。以下只检查输入结构，不根据完整目标挑选特征。

In [ ]:
audit = pd.DataFrame(
    {
        "dtype": train.dtypes.astype(str),
        "missing_rate": train.isna().mean(),
        "nunique": train.nunique(),
    }
)
audit.sort_values("missing_rate", ascending=False).head(15)

## 划分后再探索目标规律
开发集用于探索；预留集留到模型方案选好之后。

In [ ]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=["Id", "SalePrice"])
y = np.log(train["SalePrice"])
X_dev, X_hold, y_dev, y_hold = train_test_split(X, y, test_size=0.2, random_state=42)
eda = X_dev.assign(SalePrice=np.exp(y_dev))
eda.groupby("Neighborhood").agg(
    n=("SalePrice", "size"), median_price=("SalePrice", "median")
)

## 检查特征工程
只按当前行和数据字典生成变量；中位数、均值、类别编码由 Pipeline 在训练折内学习。

In [ ]:
from cases.house_prices.train import make_features

make_features(X_dev)[["TotalSF", "TotalBath", "HouseAge", "LotFrontageMissing"]].head()

## 运行完整模型比较和最终预测
本步骤重新读取数据并使用同一固定划分，运行五折 CV、预留评估和全部标签重训。首次运行后避免反复按预留分数调参。

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "cases.house_prices.train", "--data", str(DATA)], check=True
)
pd.read_csv("outputs/house_prices/cv_scores.csv")

## 检查结果
本地分数不等于排行榜分数。误差分析后的修改需要新的独立验证证据。

In [ ]:
import json

print(json.loads(Path("outputs/house_prices/summary.json").read_text()))
pd.read_csv("outputs/house_prices/holdout_errors.csv").head(10)

In [ ]:
submission = pd.read_csv("outputs/house_prices/submission.csv")
assert submission["Id"].equals(test["Id"])
assert submission["SalePrice"].gt(0).all()
submission.head()